# Curse of Dimensionality 
1. Exponential growth of space (cells table)
2. Sparsity: how big must a neighbourhood be to capture 10% of the data?
3. Volume concentrates in the corners / shell
4. Distances lose meaning (min/max ratio, relative contrast)
5. Effect on KNN: adding useless features hurts accuracy
6. Remedies: feature selection, PCA (via SVD), more data, L2 regularisation
7. Faces-style example: 10,000 features, 200 samples

In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('plots', exist_ok=True)
rng = np.random.default_rng(1)

## 1. Exponential growth of space
Split each feature into 10 intervals: cells = 10^d.

In [ ]:
print(f"{'dims':>5s} {'cells':>12s}")
for d in [1, 2, 3, 5, 10, 100]:
    print(f"{d:5d} {'10^%d' % d:>12s}   ({10**d:,})" if d <= 10 else f"{d:5d} {'10^%d' % d:>12s}")

# fraction of cells occupied by 100 samples
ds = np.arange(1, 9)
occ = []
for d in ds:
    pts = rng.random((100, d)); cells = {tuple((p * 10).astype(int)) for p in pts}
    occ.append(len(cells) / 10 ** d)
plt.figure(figsize=(5.5, 4)); plt.semilogy(ds, occ, 'o-')
plt.xlabel("dimensions"); plt.ylabel("fraction of cells occupied"); plt.title("100 samples cover less and less space")
plt.tight_layout(); plt.savefig('plots/01_cell_occupancy.png', dpi=110); plt.close()

## 2. Sparsity: neighbourhood size
To capture a fraction f of uniformly distributed data inside a hypercube
neighbourhood, its edge length is e = f^(1/d). At d=100, e=0.98: the
"local" neighbourhood is essentially the whole range of every feature.

In [ ]:
f = 0.10
dd = np.arange(1, 101); edge = f ** (1 / dd)
print("Edge length to capture 10%% of data: d=1:%.2f d=2:%.2f d=10:%.2f d=100:%.2f" % (edge[0], edge[1], edge[9], edge[99]))
plt.figure(figsize=(5.5, 4)); plt.plot(dd, edge); plt.xlabel("dimensions"); plt.ylabel("edge length (range = 1)")
plt.title("Neighbourhood needed for 10% of data"); plt.tight_layout(); plt.savefig('plots/02_neighbourhood_edge.png', dpi=110); plt.close()

## 3. Volume concentrates in the shell
Fraction of a unit ball's volume within the outer 5% radius: 1 - 0.95^d.
Also the ratio (volume of inscribed ball)/(volume of cube) -> 0.
We verify by Monte Carlo.

In [ ]:
from math import gamma, pi
def ball_volume(d, r=1.0): return pi ** (d / 2) / gamma(d / 2 + 1) * r ** d
dims = np.arange(1, 21)
ratio = [ball_volume(d, .5) / 1.0 for d in dims]          # ball radius .5 in unit cube
shell = 1 - 0.95 ** dims
# Monte Carlo check at d=8
mc = np.mean(np.linalg.norm(rng.random((200000, 8)) - .5, axis=1) <= .5)
print(f"Ball/cube volume at d=8: exact={ball_volume(8, .5):.4f}  monte-carlo={mc:.4f}")
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(dims, ratio, 'o-'); ax[0].set_title("Inscribed ball / cube volume"); ax[0].set_xlabel("dimensions")
ax[1].plot(dims, shell, 'o-'); ax[1].set_title("Share of ball volume in outer 5% shell"); ax[1].set_xlabel("dimensions")
plt.tight_layout(); plt.savefig('plots/03_volume_concentration.png', dpi=110); plt.close()

## 4. Distances lose meaning
For n random points, compare the nearest and farthest distance from a query.
As d grows, (d_max - d_min)/d_min -> 0: everyone is about equally far.

In [ ]:
def pairwise_sq(A, B):
    return (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None] - 2 * A @ B.T

dims_list = [1, 2, 5, 10, 20, 50, 100, 500, 1000]
contrast, ratio_mm = [], []
for d in dims_list:
    X = rng.random((500, d)); q = rng.random((20, d))
    D = np.sqrt(np.maximum(pairwise_sq(q, X), 0))
    contrast.append(np.mean((D.max(1) - D.min(1)) / D.min(1))); ratio_mm.append(np.mean(D.min(1) / D.max(1)))
for d, c, r in zip(dims_list, contrast, ratio_mm): print(f"d={d:5d}  relative contrast={c:8.3f}  min/max={r:.3f}")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].semilogx(dims_list, contrast, 'o-'); ax[0].set_title("(dmax-dmin)/dmin"); ax[0].set_xlabel("dimensions")
ax[1].semilogx(dims_list, ratio_mm, 'o-'); ax[1].set_title("dmin/dmax -> 1"); ax[1].set_xlabel("dimensions")
for d, c in [(2, 'C0'), (50, 'C1'), (1000, 'C2')]:
    X = rng.random((300, d)); D = np.sqrt(np.maximum(pairwise_sq(X, X), 0))[np.triu_indices(300, 1)]
    ax[2].hist(D / D.mean(), bins=40, alpha=.5, color=c, label=f"d={d}", density=True)
ax[2].set_title("Pairwise distance / mean"); ax[2].legend()
plt.tight_layout(); plt.savefig('plots/04_distance_concentration.png', dpi=110); plt.close()

## 5. Effect on KNN
2 informative features + k pure-noise features. KNN accuracy decays toward
chance even though the informative signal never changed.

In [ ]:
def knn_predict(Xtr, ytr, Xte, k=5):
    D = pairwise_sq(Xte, Xtr); nn = np.argpartition(D, k, axis=1)[:, :k]
    return (ytr[nn].mean(1) > .5).astype(int)

def make_data(n, n_noise, rng):
    y = rng.integers(0, 2, n)
    inf = rng.normal(0, 1, (n, 2)) + y[:, None] * 1.8
    return np.hstack([inf, rng.normal(0, 1, (n, n_noise))]), y

noise_dims = [0, 1, 2, 5, 10, 20, 50, 100, 200, 500]
acc_knn = []
for nz in noise_dims:
    accs = []
    for rep in range(3):
        Xtr, ytr = make_data(200, nz, rng); Xte, yte = make_data(400, nz, rng)
        accs.append(np.mean(knn_predict(Xtr, ytr, Xte) == yte))
    acc_knn.append(np.mean(accs))
print("KNN acc vs #noise dims:", dict(zip(noise_dims, np.round(acc_knn, 3))))
plt.figure(figsize=(6, 4)); plt.semilogx(np.array(noise_dims) + 1, acc_knn, 'o-'); plt.axhline(.5, color='k', ls=':')
plt.xlabel("useless features added (+1)"); plt.ylabel("KNN accuracy"); plt.title("KNN degrades with irrelevant dimensions")
plt.tight_layout(); plt.savefig('plots/05_knn_vs_dims.png', dpi=110); plt.close()

# more data helps
sizes = [50, 100, 200, 500, 1000, 3000]; acc_n = []
for n in sizes:
    Xtr, ytr = make_data(n, 20, rng); Xte, yte = make_data(400, 20, rng)
    acc_n.append(np.mean(knn_predict(Xtr, ytr, Xte) == yte))
print("KNN acc (22 dims) vs training size:", dict(zip(sizes, np.round(acc_n, 3))))

## 6. Remedies
### 6a. Feature selection (filter: |correlation| with label)
### 6b. PCA from scratch with SVD
### 6c. Ridge (L2) regularisation for linear models when d > n

In [ ]:
def select_k_best(X, y, k):
    """Filter method: keep the k features with highest |corr(x_j, y)|."""
    Xc = (X - X.mean(0)) / (X.std(0) + 1e-12); yc = (y - y.mean()) / y.std()
    return np.argsort(-np.abs(Xc.T @ yc) / len(y))[:k]

class PCA:
    """Principal components via SVD of the centred data matrix."""
    def __init__(self, n_components): self.k = n_components
    def fit(self, X):
        self.mean_ = X.mean(0); U, S, Vt = np.linalg.svd(X - self.mean_, full_matrices=False)
        self.components_ = Vt[:self.k]; self.explained_ = S ** 2 / (len(X) - 1); return self
    def transform(self, X): return (X - self.mean_) @ self.components_.T
    def inverse_transform(self, Z): return Z @ self.components_ + self.mean_

def ridge_fit(X, y, lam):
    """w = (X^T X + lam I)^-1 X^T y, via the dual form when d > n (cheaper)."""
    Xc = np.column_stack([np.ones(len(X)), X]); n, d = Xc.shape
    reg = lam * np.eye(d); reg[0, 0] = 0
    if d > n:  # dual: w = X^T (X X^T + lam I)^-1 y
        w = Xc.T @ np.linalg.solve(Xc @ Xc.T + lam * np.eye(n) + 1e-9 * np.eye(n), y)
    else: w = np.linalg.solve(Xc.T @ Xc + reg, Xc.T @ y)
    return w
def ridge_predict(w, X): return np.column_stack([np.ones(len(X)), X]) @ w

Xtr, ytr = make_data(200, 200, rng); Xte, yte = make_data(400, 200, rng)
acc_all = np.mean(knn_predict(Xtr, ytr, Xte) == yte)
sel = select_k_best(Xtr, ytr, 5)
acc_sel = np.mean(knn_predict(Xtr[:, sel], ytr, Xte[:, sel]) == yte)
pca = PCA(2).fit(Xtr); acc_pca = np.mean(knn_predict(pca.transform(Xtr), ytr, pca.transform(Xte)) == yte)
sel_true = sorted(sel[sel < 2].tolist())
print(f"KNN on 202 dims={acc_all:.3f} | select 5 best={acc_sel:.3f} (found informative {sel_true}) | PCA(2)={acc_pca:.3f}")

# ridge vs OLS with d > n: labels as +/-1 regression
ytr_s, yte_s = 2 * ytr - 1., 2 * yte - 1.
for lam in [1e-6, 1, 100, 1000]:
    w = ridge_fit(Xtr, ytr_s, lam)
    print(f"ridge lam={lam:>8}: test acc={np.mean((ridge_predict(w, Xte) > 0) == yte):.3f}")

## 7. Faces-style example
200 "images" of 100x100=10,000 pixels, but the class signal lives on a
handful of directions (latent factors). PCA to a few components recovers it.

In [ ]:
n, d, latent = 200, 10000, 5
L = rng.normal(size=(latent, d))                       # pixel patterns for latent factors
def faces(n):
    y = rng.integers(0, 2, n); Z = rng.normal(size=(n, latent)); Z[:, 0] += 1.5 * (2 * y - 1)
    return Z @ L + rng.normal(0, 3, (n, d)), y
Ftr, fytr = faces(200); Fte, fyte = faces(300)
acc_raw = np.mean(knn_predict(Ftr, fytr, Fte) == fyte)
pca = PCA(10).fit(Ftr)
acc_p = np.mean(knn_predict(pca.transform(Ftr), fytr, pca.transform(Fte)) == fyte)
print(f"10,000-D raw KNN acc={acc_raw:.3f} | PCA(10) KNN acc={acc_p:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ev = pca.explained_ / (np.linalg.svd(Ftr - Ftr.mean(0), compute_uv=False) ** 2 / (len(Ftr) - 1)).sum()
ax[0].bar(range(1, 11), ev[:10]); ax[0].set_title("Explained variance ratio (PCA)"); ax[0].set_xlabel("component")
Zp = pca.transform(Ftr); ax[1].scatter(Zp[:, 0], Zp[:, 1], c=fytr, cmap='coolwarm', s=15); ax[1].set_title("First 2 PCs of 10,000-D data")
plt.tight_layout(); plt.savefig('plots/06_pca_faces.png', dpi=110); plt.close()

# Optional sklearn PCA check (subspace agreement)
try:
    from sklearn.decomposition import PCA as SkPCA
    sk = SkPCA(2).fit(Xtr)
    print("sklearn PCA explained var ratio:", np.round(sk.explained_variance_ratio_, 4),
          "| ours:", np.round(PCA(2).fit(Xtr).explained_[:2] / np.var(Xtr, 0, ddof=1).sum(), 4))
except Exception as e:
    print("sklearn check skipped:", e)